# 33 RL evaluation (CPU runtime)

Scores the RL model on validation, dev and mini-dev, and compares it on validation with the SFT model it started from and the kept DPO model, by the same rule as before: among the models whose validation EX is not significantly below the best (paired McNemar test), the highest validation R-VES is kept. Dev and mini-dev are reported, never used to choose.

**Runtime > Change runtime type > CPU** (timing runs on the same machine type as all earlier timing).

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = False  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
import json, os
EXPORT = "sft_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# RL starts from the SFT model kept on validation by R-VES (25_sft_evaluate: method, then epoch 1 or 2).
INIT = json.load(open(f"{R}/{EXPORT}_epoch_choice_rves/selection.json"))["winner"]
PILOT, RUN = "rl_v1_pilot", "rl_v1"
print("starts from:", INIT, "| pilot run:", PILOT, "| full run:", RUN)
MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
METHOD = "rl-" + json.load(open(f"{R}/{RUN}/adapter_meta.json"))["arm"]
DPO = "dpo_v1_best"  # the DPO model kept on validation (31_dpo_evaluate)
meta = json.load(open(f"{R}/{EXPORT}/export_meta.json"))
VAL_DBS = ",".join(meta["holdout_db_ids"])

### 1. Stage the validation databases and the dev databases

In [ ]:
!effsql stage --split train --db-ids {VAL_DBS}
!effsql stage --split dev

### 2. Score the RL model (resumable)

In [ ]:
!python -m effsql.train eval-val --run {RUN}
!effsql evaluate --pred {R}/{RUN}/predictions_dev.jsonl --split dev --timing none --run {RUN}_dev --model {MODEL} --method {METHOD}
!effsql evaluate --pred {R}/{RUN}/predictions_minidev.jsonl --split minidev --timing quick --run {RUN}_minidev --model {MODEL} --method {METHOD}

### 3. RL against its SFT starting point and the DPO model on validation

In [ ]:
!python -m effsql.train select --runs {INIT},{DPO},{RUN} --out rl_v1_choice

### 4. Training log: per iteration, on the answers the model wrote while training

In [ ]:
from effsql.train.rl import format_summary
print(format_summary(json.load(open(f"{R}/{RUN}/rl_summary.json"))))

### 5. All results: validation, dev and mini-dev, each model against the SFT starting point

Main metric: **R-VES** (BIRD's reward per question with step counts in place of seconds: at least 2x faster than gold 112, 1 to 2x 100, up to 2x slower 87, 2 to 4x slower 71, slower 50, wrong 0; gold = 100), with a 95% bootstrap interval. **R-VES/EX**: the same over correct answers only, the efficiency apart from accuracy. **VES** (100 x square root of the speed-up, unbounded) is shown but a few extreme questions drive it. Then each model against the reference on the same questions: R-VES difference (Wilcoxon test), EX (McNemar test), and how often its query is cheaper or costlier by at least 10% when both are correct (sign test). Nothing is executed: the table reads the saved evaluations. Saved as `report.txt` and `report.json`.

In [ ]:
REPORT = [INIT, "pool_v1_{split}_base", f"{EXPORT}_gold_e1", DPO, RUN]
LABELS = ",".join(["pool_v1_{split}_base=zero-shot"] + [f"{r}={r.replace(EXPORT + '_', '')}" for r in REPORT if "{split}" not in r])
REF, OUT = INIT, "rl_v1_report"
!python -m effsql.train report --runs "{','.join(REPORT)}" --ref {REF} --labels "{LABELS}" --out {OUT}

### Done: disconnect the runtime